# Loss and overlap metric

How the training loss (`src/losses.py`) and the Dice metric (`src/evaluation.py`) behave,
on small synthetic 2D masks with a fixed seed:

1. A toy ground truth and a toy prediction
2. Overlap: true / false positives and negatives, Dice and IoU
3. From the Dice metric to the Dice **loss** (soft Dice on probabilities)
4. Binary cross-entropy per pixel
5. Why both: a small object and an empty prediction
6. How the loss changes as the prediction moves away from the target

The same functions are used as in training; only the images are 2D instead of 3D.

In [20]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
from plotly.subplots import make_subplots

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "src").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from src.losses import DiceBCELoss

SEED = 0
SIZE = 64
rng = np.random.default_rng(SEED)
STATIC = {"staticPlot": True}  # plots without zoom / hover

# colours
TP, FP, FN, TN = "#4c9a6a", "#c56d6d", "#5b7fb5", "#eeeeee"

## 1. Toy ground truth and prediction

- Ground truth: a disc of radius 12 (1 inside, 0 outside).
- Prediction: the network outputs **logits**. Here they are made by hand from a disc that is
  shifted and slightly larger, plus noise: positive inside the predicted disc, negative outside,
  with larger magnitude further from its edge. `sigmoid(logits)` gives probabilities.

In [21]:
yy, xx = np.mgrid[:SIZE, :SIZE]


def disc(cx, cy, r):
    return np.hypot(xx - cx, yy - cy) <= r


def toy_logits(cx, cy, r, sharpness=0.6, noise=0.8):
    """Signed distance to a disc edge, scaled, plus Gaussian noise."""
    
    signed = r - np.hypot(xx - cx, yy - cy)  # > 0 inside, < 0 outside
    return sharpness * signed + rng.normal(0, noise, signed.shape)


target = disc(32, 32, 12).astype(np.float32)
logits = toy_logits(36, 30, 14)
prob = 1 / (1 + np.exp(-logits))
pred = (prob > 0.5).astype(np.float32)

fig = make_subplots(
    rows=2, cols=2, horizontal_spacing=0.14, vertical_spacing=0.08,
    subplot_titles=("ground truth", "logits", "sigmoid(logits)", "prediction (p > 0.5)"),
)
fig.add_trace(go.Heatmap(z=target, colorscale="Greys", reversescale=True, showscale=False), 1, 1)
fig.add_trace(go.Heatmap(z=logits, colorscale="RdBu", zmid=0,
                         colorbar=dict(title="logit", x=1.0, y=0.78, len=0.42)), 1, 2)
fig.add_trace(go.Heatmap(z=prob, colorscale="Viridis", zmin=0, zmax=1,
                         colorbar=dict(title="p", x=0.44, xanchor="left", y=0.22, len=0.42)), 2, 1)
fig.add_trace(go.Heatmap(z=pred, colorscale="Greys", reversescale=True, showscale=False), 2, 2)

fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False, autorange="reversed")
for i in range(1, 5):
    sx = "" if i == 1 else str(i)
    fig.layout[f"yaxis{sx}"].update(scaleanchor=f"x{sx}")

fig.update_layout(width=750, height=760, plot_bgcolor="white", margin=dict(t=40, b=10, l=10, r=10))
fig.show(config=STATIC)

## 2. Overlap and the Dice metric

Comparing the binary prediction with the ground truth, every pixel is one of:

| | ground truth 1 | ground truth 0 |
|---|---|---|
| **predicted 1** | true positive (TP) | false positive (FP) |
| **predicted 0** | false negative (FN) | true negative (TN) |

$$\text{Dice} = \frac{2\,TP}{2\,TP + FP + FN} = \frac{2|A \cap B|}{|A| + |B|}
\qquad
J = \frac{TP}{TP + FP + FN} = \frac{|A \cap B|}{|A \cup B|}$$

$J$ is the Jaccard index. The two are directly related, $\text{Dice} = \frac{2J}{1 + J}$,
so they always rank predictions the same way; Jaccard gives lower numbers for the same overlap.

TN does not appear in either formula, so the large background does not inflate the score.
Both are 1 for a perfect overlap and 0 for no overlap. BraTS reports Dice (per region),
which is what `evaluate_dice` computes.

In [22]:
def overlap_counts(pred, target):
    p, t = pred.astype(bool), target.astype(bool)
    return {"TP": int((p & t).sum()), "FP": int((p & ~t).sum()),
            "FN": int((~p & t).sum()), "TN": int((~p & ~t).sum())}


def dice_jaccard(c):
    denom = 2 * c["TP"] + c["FP"] + c["FN"]
    dice = 2 * c["TP"] / denom if denom else 1.0
    jaccard = c["TP"] / (c["TP"] + c["FP"] + c["FN"]) if denom else 1.0
    return dice, jaccard


counts = overlap_counts(pred, target)
dice, jaccard = dice_jaccard(counts)

# 0 = TN, 1 = FP, 2 = FN, 3 = TP
cat = np.zeros_like(target, dtype=int)
p, t = pred.astype(bool), target.astype(bool)
cat[p & ~t], cat[~p & t], cat[p & t] = 1, 2, 3
colorscale = [[0, TN], [0.25, TN], [0.25, FP], [0.5, FP], [0.5, FN], [0.75, FN], [0.75, TP], [1, TP]]

fig = go.Figure(go.Heatmap(z=cat, zmin=-0.5, zmax=3.5, colorscale=colorscale, showscale=False))
for name, colour in (("true positive", TP), ("false positive", FP), ("false negative", FN)):
    fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", name=name,
                             marker=dict(symbol="square", size=14, color=colour)))
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False, autorange="reversed", scaleanchor="x")
fig.update_layout(width=520, height=520, plot_bgcolor="white", margin=dict(t=50, b=10, l=10, r=10),
                  title=dict(text=f"<b>Overlap</b>   Dice = {dice:.3f}   Jaccard = {jaccard:.3f}", x=0.5),
                  legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.02, yanchor="top"))
fig.show(config=STATIC)
pd.DataFrame([counts])

,TP,FP,FN,TN
0,392,224,49,3431


## 3. From the Dice metric to the Dice loss

The Dice metric uses a **binary** prediction (`p > 0.5`). Thresholding has no useful gradient,
so the network cannot learn from it directly. The loss uses the **probabilities** instead
(*soft Dice*). With `squared_pred=True`, as in `src/losses.py`:

$$\mathcal{L}_{\text{Dice}} = 1 - \frac{2\sum_i p_i\, g_i}{\sum_i p_i^2 + \sum_i g_i^2 + \varepsilon}$$

where $p_i$ is the predicted probability and $g_i \in \{0, 1\}$ the ground truth of pixel $i$.
For a perfect, fully confident prediction the loss is 0. Below, the formula computed by hand
is compared with the Dice part of `DiceBCELoss`.

In [23]:
loss_fn = DiceBCELoss()
logits_t = torch.tensor(logits, dtype=torch.float32)[None, None]  # (B=1, C=1, H, W)
target_t = torch.tensor(target)[None, None]

soft_dice_manual = 1 - 2 * (prob * target).sum() / ((prob ** 2).sum() + (target ** 2).sum() + 1e-5)
pd.DataFrame({
    "value": [
        1 - dice,
        soft_dice_manual,
        loss_fn.dice(logits_t, target_t).item(),
    ],
}, index=["1 − Dice metric (binary prediction)",
          "soft Dice loss, by hand",
          "soft Dice loss, DiceBCELoss.dice"]).round(4)

,value
1 − Dice metric (binary prediction),0.2583
"soft Dice loss, by hand",0.2052
"soft Dice loss, DiceBCELoss.dice",0.2052


The soft Dice loss is close to `1 − Dice` but not equal: uncertain pixels (probability near 0.5)
count only partly, so the loss keeps pushing the network to be confident and correct.

## 4. Binary cross-entropy per pixel

$$\mathcal{L}_{\text{BCE}} = -\frac{1}{N}\sum_i \big[g_i \log p_i + (1 - g_i)\log(1 - p_i)\big]$$

Unlike Dice, BCE is a sum of **independent per-pixel** terms. The map shows which pixels
contribute most: confident mistakes (high probability where the ground truth is 0, or the
reverse) are penalised strongly.

In [24]:
eps = 1e-7
bce_map = -(target * np.log(prob + eps) + (1 - target) * np.log(1 - prob + eps))

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.08,
                    subplot_titles=("overlap (from section 2)", "BCE per pixel"))
fig.add_trace(go.Heatmap(z=cat, zmin=-0.5, zmax=3.5, colorscale=colorscale, showscale=False), 1, 1)
fig.add_trace(go.Heatmap(z=bce_map, colorscale="Reds", colorbar=dict(title="BCE")), 1, 2)
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False, autorange="reversed")
fig.layout.yaxis.update(scaleanchor="x")
fig.layout.yaxis2.update(scaleanchor="x2")
fig.update_layout(width=900, height=430, plot_bgcolor="white", margin=dict(t=40, b=10, l=10, r=10))
fig.show(config=STATIC)

print(f"mean BCE by hand:       {bce_map.mean():.4f}")
print(f"DiceBCELoss.bce:        {loss_fn.bce(logits_t, target_t).item():.4f}")

mean BCE by hand:       0.1494
DiceBCELoss.bce:        0.1494


## 5. Why both terms: a small object

Tumour regions are often a tiny fraction of the volume. Below, the target is a small disc
(radius 3), and three predictions are compared, including one that predicts **nothing**.

- BCE averages over all pixels. When almost all pixels are background, predicting "nothing"
  is right for most of them, so BCE stays small.
- Dice ignores true negatives, so an empty prediction of a non-empty target gets Dice 0
  and Dice loss 1.

The two terms therefore complement each other: Dice focuses on the overlap of the object,
BCE gives a smooth per-pixel signal everywhere.

In [25]:
small = disc(32, 32, 3).astype(np.float32)
# no noise here, so the shapes are exact copies of the target (shifted or not)
cases = {
    "predicts nothing": np.full((SIZE, SIZE), -6.0),
    "shifted 3 px right": toy_logits(35, 32, 3, sharpness=2.0, noise=0.0),
    "correct": toy_logits(32, 32, 3, sharpness=2.0, noise=0.0),
}

small_t = torch.tensor(small)[None, None]
rows, panels = [], []
for name, lg in cases.items():
    lg_t = torch.tensor(lg, dtype=torch.float32)[None, None]
    pr = (1 / (1 + np.exp(-lg)) > 0.5).astype(np.float32)
    d, _ = dice_jaccard(overlap_counts(pr, small)) 
    rows.append({"case": name, "Dice metric": d,
                 "Dice loss": loss_fn.dice(lg_t, small_t).item(),
                 "BCE": loss_fn.bce(lg_t, small_t).item(),
                 "total loss": loss_fn(lg_t, small_t).item()})
    c = np.zeros_like(small, dtype=int)
    p, t = pr.astype(bool), small.astype(bool)
    c[p & ~t], c[~p & t], c[p & t] = 1, 2, 3
    panels.append((name, c))

fig = make_subplots(rows=1, cols=3, subplot_titles=[n for n, _ in panels], horizontal_spacing=0.04)
for i, (_, c) in enumerate(panels, start=1):
    fig.add_trace(go.Heatmap(z=c[24:41, 24:44], zmin=-0.5, zmax=3.5, colorscale=colorscale,
                             showscale=False), 1, i)
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False, autorange="reversed")
for i in range(1, 4):
    sx = "" if i == 1 else str(i)
    fig.layout[f"yaxis{sx}"].update(scaleanchor=f"x{sx}")
fig.update_layout(width=900, height=330, plot_bgcolor="white", margin=dict(t=40, b=10, l=10, r=10))
fig.show(config=STATIC)

pd.DataFrame(rows).set_index("case").round(4)

,Dice metric,Dice loss,BCE,total loss
case,,,,
predicts nothing,0.0000,0.9951,0.0450,1.0400
shifted 3 px right,0.4074,0.5373,0.0234,0.5606
correct,0.9259,0.0797,0.0040,0.0837


## 6. Loss as the prediction moves away from the target

The predicted disc is shifted step by step away from the ground truth (same radius, same noise
seed). The Dice metric falls in steps (it counts whole pixels), while the soft Dice loss and BCE
change smoothly — this smoothness is what makes them usable for gradient descent.

In [26]:
shifts = np.arange(0, 27)
records = []
for s in shifts:
    rng = np.random.default_rng(SEED)  # same noise at every shift
    lg = toy_logits(32 + s, 32, 12)
    lg_t = torch.tensor(lg, dtype=torch.float32)[None, None]
    pr = (1 / (1 + np.exp(-lg)) > 0.5).astype(np.float32)
    d, _ = dice_jaccard(overlap_counts(pr, small)) 
    
    records.append({"shift": s, "Dice metric": d,
                    "Dice loss": loss_fn.dice(lg_t, target_t).item(),
                    "BCE": loss_fn.bce(lg_t, target_t).item(),
                    "total loss": loss_fn(lg_t, target_t).item()})
curves = pd.DataFrame(records)

fig = go.Figure()
styles = {"Dice metric": ("#4c9a6a", "solid"), "Dice loss": ("#c56d6d", "solid"),
          "BCE": ("#5b7fb5", "dash"), "total loss": ("#444444", "dot")}
for col, (colour, dash) in styles.items():
    fig.add_trace(go.Scatter(x=curves["shift"], y=curves[col], name=col, mode="lines",
                             line=dict(color=colour, dash=dash, width=3)))
fig.add_vline(x=24, line=dict(color="#999", dash="dot"),
              annotation_text="discs stop overlapping", annotation_position="top left")
fig.update_layout(width=850, height=450, template="simple_white",
                  xaxis_title="shift of the predicted disc (pixels)", yaxis_title="value",
                  legend=dict(orientation="h", x=0.5, xanchor="center", y=1.08),
                  margin=dict(t=60, b=50, l=60, r=20))
fig.show(config=STATIC)